In [0]:
import requests

import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# 🔐 GET FROM DATABRICKS SECRETS (FIX)

BASE_URL = dbutils.secrets.get(scope="slainte-secrets", key="supabase-url").strip()

API_KEY  = dbutils.secrets.get(scope="slainte-secrets", key="supabase-api-key").strip()

HEADERS  = {"apikey": API_KEY, "Authorization": f"Bearer {API_KEY}"}

def fetch(table):

    df = pd.DataFrame(requests.get(f"{BASE_URL}/{table}", headers=HEADERS).json())

    return df.where(lambda x: x.notna(), None)

# Fetch

fm = fetch("field_mappings")

ji = fetch("jira_integrations")

# To Spark

df_fm = spark.createDataFrame(fm).withColumn("integration_id", F.trim(F.col("integration_id").cast("string")))

df_ji = spark.createDataFrame(ji).withColumn("id", F.trim(F.col("id").cast("string")))

# Join to get user_id (drop project_key from df_ji to avoid duplicate)

df = df_fm.join(

    df_ji.select(F.col("id").alias("integration_id"), "user_id"),

    on="integration_id",

    how="left"

).withColumn("ingestion_timestamp", F.current_timestamp())

# Overwrite

TARGET = "workspace.slainte_bronze.supabase_field_mappings_raw"

df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(TARGET)

print("✅ Saved:", TARGET)

print("Rows:", df.count())
 